In [1]:
# ============================================================
# ANALISIS MODULAR POR DATASET (sheet: "newmodels")
# 1) FIDELIDAD (FI)   : 0.45*PCC + 0.35*SC + 0.20*(1 - RMSE_norm_dataset)
# 2) COMPLEJIDAD (CI) : min-max por dataset sobre log10(lat, MACs, params)
#                       con pesos adaptativos si latencia ~ MACs
# 3) RELACION FI–CI   : factibilidad, Pareto, TOP-K y gráficas por dataset
# Salidas: PNGs por dataset + CSVs agregados
# ============================================================

import numpy as np
import pandas as pd# ============================================================
# Paper-ready plots para RMSE, PCC, SC con FI geométrico
# - RMSE -> utilidad [0,1] (skill score / NRMSE robusto)
# - FI = GeometricMean( PCC^w1, SC^w2, u_RMSE^w3 )
# - Scatter RMSE–PCC (size=SC, color=FI) + Frente de Pareto + labels con halo
# - Barras por métrica con error, Average rank, Heatmap, Radar
# - Exporta PDF (vectorial) + PNG
# ============================================================

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as pe

# ========= Estilo general (paper) =========
plt.style.use('default')
sns.set_palette("colorblind")
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.figsize': (10, 6),
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.transparent': True
})

# ========= Parámetros =========
ROBUST = True
Q_LOW, Q_HIGH = 0.05, 0.95
# Pesos recomendados (balanceados, puedes ajustarlos)
W_PCC, W_SC, W_RMSE = 0.40, 0.30, 0.30
TOP_K = 10
OUTPUT_DIR = "figs_paper"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========= Utilidades =========
def robust_minmax(series, q_low=Q_LOW, q_high=Q_HIGH):
    """Min–max robusto (Q05–Q95). Si rango inválido -> 0.5."""
    s = pd.to_numeric(series, errors='coerce').astype(float).copy()
    lo, hi = s.quantile(q_low), s.quantile(q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.full(len(s), 0.5), index=s.index)
    out = (s - lo) / (hi - lo)
    return out.clip(0, 1)

def rmse_to_utility(g: pd.DataFrame, mode='skill', baseline_col='RMSE_baseline'):
    """
    Transforma RMSE_mean -> utilidad [0,1], 1=mejor.
    - 'skill': u = max(0, 1 - RMSE/RMSE_baseline)
    - 'nrmse_sigma': RMSE / y_sigma -> minmax robusto -> u=1-n
    - fallback: minmax robusto por dataset -> u=1-n
    """
    rmse = pd.to_numeric(g['RMSE_mean'], errors='coerce')
    if mode == 'skill' and baseline_col in g.columns and g[baseline_col].notna().any():
        base = pd.to_numeric(g[baseline_col], errors='coerce').replace(0, np.nan)
        u = 1 - (rmse / base)
        u = u.clip(lower=0)   # No permitir negativos si peor que baseline
        u = u.clip(0, 1)
        return u.fillna(0.0)
    elif mode == 'nrmse_sigma' and 'y_sigma' in g.columns and g['y_sigma'].notna().any():
        nrmse = rmse / pd.to_numeric(g['y_sigma'], errors='coerce').replace(0, np.nan)
        n = robust_minmax(nrmse)
        return (1 - n).fillna(0.0)
    else:
        n = robust_minmax(rmse)
        return (1 - n).fillna(0.0)

def rank_block_by_dataset(g: pd.DataFrame) -> pd.DataFrame:
    """Rangos por métrica (1=mejor) dentro del dataset."""
    out = g.copy()
    out['rank_RMSE'] = out['RMSE_mean'].rank(ascending=True,  method='min')
    out['rank_PCC']  = out['CC_mean'].rank(ascending=False, method='min')
    out['rank_SC']   = out['SC_mean'].rank(ascending=False, method='min')
    out['avg_rank']  = out[['rank_RMSE','rank_PCC','rank_SC']].mean(axis=1)
    return out

def add_fidelity_by_dataset(frame: pd.DataFrame, rmse_mode='skill') -> pd.DataFrame:
    """
    FI por dataset usando media geométrica ponderada:
      FI = (PCC^w1 * SC^w2 * u_RMSE^w3)^(1/(w1+w2+w3)), con clamp [0,1].
    """
    out = []
    for dname, g in frame.groupby('Dataset', sort=False):
        g = g.copy()
        g['CC_mean'] = g['CC_mean'].clip(0, 1)
        g['SC_mean'] = g['SC_mean'].clip(0, 1)
        g['u_RMSE']  = rmse_to_utility(g, mode=rmse_mode)
        eps = 1e-6
        p = (g['CC_mean'].clip(0,1)+eps)**W_PCC
        s = (g['SC_mean'].clip(0,1)+eps)**W_SC
        r = (g['u_RMSE'].clip(0,1)+eps)**W_RMSE
        g['FI'] = (p * s * r)**(1.0/(W_PCC+W_SC+W_RMSE))
        out.append(g)
    return pd.concat(out, ignore_index=True)

def pareto_front(df):
    """
    Marca puntos no dominados en (↑PCC, ↑SC, ↓RMSE).
    Retorna máscara booleana de no-dominados.
    """
    vals = df[['RMSE_mean','CC_mean','SC_mean']].values
    n = len(vals)
    is_nd = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_nd[i]:
            continue
        for j in range(n):
            if i == j:
                continue
            # j domina a i si PCC_j>=, SC_j>=, RMSE_j<= y mejora estricta en alguna
            if (vals[j,1] >= vals[i,1]) and (vals[j,2] >= vals[i,2]) and (vals[j,0] <= vals[i,0]) and \
               ((vals[j,1] > vals[i,1]) or (vals[j,2] > vals[i,2]) or (vals[j,0] < vals[i,0])):
                is_nd[i] = False
                break
    return is_nd

# ========= Plots =========
def _save(fig_name_base):
    pdf = os.path.join(OUTPUT_DIR, f"{fig_name_base}.pdf")
    png = os.path.join(OUTPUT_DIR, f"{fig_name_base}.png")
    plt.savefig(pdf); plt.savefig(png)

def plot_fi_top(g, dname, top_k=TOP_K):
    """Leaderboard FI — Top-K (PDF+PNG)."""
    top = (g.sort_values('FI', ascending=False)
             .head(top_k)
             [['Variant','Type','FI','CC_mean','u_RMSE','SC_mean']]
             .reset_index(drop=True))
    plt.figure(figsize=(10, 5.2))
    order = top.sort_values('FI', ascending=True)
    plt.barh(order['Variant'], order['FI'], alpha=0.95)
    plt.xlabel('Fidelity Index (0–1)')
    plt.title(f'{dname}: Top-{top_k} Fidelity Index')
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout(); _save(f'fi_top_{dname}'); plt.show()
    return top

def plot_scatter_rmse_pcc(g, dname, top_labels):
    """RMSE vs PCC; size=SC, color=FI, Pareto y etiquetas con halo."""
    plt.figure(figsize=(8.8, 6.4))
    sc = plt.scatter(g['RMSE_mean'], g['CC_mean'],
                     s=360*g['SC_mean'].fillna(0.5)+40,
                     c=g['FI'], cmap='viridis', alpha=0.85,
                     edgecolors='k', linewidths=0.6)
    cbar = plt.colorbar(sc); cbar.set_label('Fidelity Index')
    plt.xlabel('RMSE (lower is better)'); plt.ylabel('PCC (higher is better)')
    plt.title(f'{dname}: RMSE vs PCC (bubble=SC, color=FI)')

    # Etiquetas Top-K con halo blanco (mejor legibilidad)
    lab = g.merge(top_labels[['Variant']], on='Variant', how='inner')
    for _, r in lab.iterrows():
        txt = plt.text(float(r['RMSE_mean']), float(r['CC_mean']), str(r['Variant']),
                       fontsize=10, weight='bold', ha='center', va='center')
        txt.set_path_effects([pe.withStroke(linewidth=3, foreground="white")])

    # Frente de Pareto
    nd = pareto_front(g)
    plt.scatter(g.loc[nd, 'RMSE_mean'], g.loc[nd, 'CC_mean'],
                s=110, facecolors='none', edgecolors='red', linewidths=1.6,
                label='Pareto front')
    plt.legend(frameon=False)
    plt.grid(True, alpha=0.3)
    plt.tight_layout(); _save(f'scatter_{dname}'); plt.show()

def plot_metric_bars_with_error(g, dname, top_k=TOP_K):
    """Barras con error para RMSE, PCC y SC (Top-K por FI)."""
    top = g.sort_values('FI', ascending=False).head(top_k)
    order = top.sort_values('RMSE_mean', ascending=True)['Variant']  # orden por RMSE

    # RMSE
    plt.figure(figsize=(10,5))
    y = top.set_index('Variant').loc[order]['RMSE_mean']
    yerr = None
    if 'RMSE_std' in top.columns and not top['RMSE_std'].isna().all():
        yerr = top.set_index('Variant').loc[order]['RMSE_std'].values
    plt.barh(y.index, y.values, xerr=yerr, alpha=0.95)
    plt.xlabel('RMSE (lower is better)'); plt.title(f'{dname}: RMSE (Top-{top_k})')
    plt.grid(True, axis='x', alpha=0.3); plt.tight_layout(); _save(f'rmse_bars_{dname}'); plt.show()

    # PCC
    plt.figure(figsize=(10,5))
    y = top.set_index('Variant').loc[order]['CC_mean']
    yerr = None
    if 'PCC_std' in top.columns and not top['PCC_std'].isna().all():
        yerr = top.set_index('Variant').loc[order]['PCC_std'].values
    plt.barh(y.index, y.values, xerr=yerr, alpha=0.95)
    plt.xlabel('PCC (higher is better)'); plt.title(f'{dname}: PCC (Top-{top_k})')
    plt.grid(True, axis='x', alpha=0.3); plt.tight_layout(); _save(f'pcc_bars_{dname}'); plt.show()

    # SC
    plt.figure(figsize=(10,5))
    y = top.set_index('Variant').loc[order]['SC_mean']
    yerr = None
    if 'SpectralCosine_std' in top.columns and not top['SpectralCosine_std'].isna().all():
        yerr = top.set_index('Variant').loc[order]['SpectralCosine_std'].values
    plt.barh(y.index, y.values, xerr=yerr, alpha=0.95)
    plt.xlabel('SC (higher is better)'); plt.title(f'{dname}: SC (Top-{top_k})')
    plt.grid(True, axis='x', alpha=0.3); plt.tight_layout(); _save(f'sc_bars_{dname}'); plt.show()

def plot_avg_rank_bars(g, dname, top_k=TOP_K):
    """Barras de promedio de rangos (menor=mejor)."""
    gr = rank_block_by_dataset(g).sort_values('avg_rank', ascending=True).head(top_k)
    plt.figure(figsize=(10,5))
    plt.barh(gr['Variant'], gr['avg_rank'], alpha=0.95)
    plt.xlabel('Average rank (RMSE, PCC, SC)'); plt.title(f'{dname}: Average rank (Top-{top_k})')
    plt.gca().invert_yaxis()
    plt.grid(True, axis='x', alpha=0.3); plt.tight_layout(); _save(f'avg_rank_{dname}'); plt.show()
    return gr[['Variant','avg_rank','rank_RMSE','rank_PCC','rank_SC']]

def plot_rank_heatmap(g, dname, top_k=TOP_K):
    """Heatmap de rangos para Top-K por FI."""
    top = g.sort_values('FI', ascending=False).head(top_k)
    ranks = rank_block_by_dataset(top)[['Variant','rank_RMSE','rank_PCC','rank_SC']].set_index('Variant')
    plt.figure(figsize=(6.5, 0.6 + 0.36*len(ranks)))
    plt.imshow(ranks.values, aspect='auto', cmap='viridis_r')
    plt.colorbar(label='Rank (1=best)')
    plt.xticks(np.arange(3), ['RMSE','PCC','SC'])
    plt.yticks(np.arange(len(ranks.index)), ranks.index)
    plt.title(f'{dname}: Metric ranks (Top-{top_k} by FI)')
    plt.tight_layout(); _save(f'rank_heatmap_{dname}'); plt.show()

def plot_radar_topk(g, dname, top_k=5):
    """Radar de fidelidad: PCC, SC, u_RMSE (Top-K por FI)."""
    from math import pi
    gg = g.copy()
    top = gg.sort_values('FI', ascending=False).head(top_k)
    metrics = ['CC_mean','SC_mean','u_RMSE']
    labels = ['PCC','SC','u(RMSE)']
    angles = [n / float(len(metrics)) * 2 * pi for n in range(len(metrics))]
    angles += angles[:1]
    plt.figure(figsize=(6.8,6.8))
    ax = plt.subplot(111, polar=True)
    ax.set_theta_offset(pi / 2); ax.set_theta_direction(-1)
    plt.xticks(angles[:-1], labels)
    ax.set_rlabel_position(0); plt.yticks([0.25,0.5,0.75], ['0.25','0.5','0.75'], color="grey", size=9); plt.ylim(0,1)
    for _, r in top.iterrows():
        vals = [float(r[m]) for m in metrics]; vals += vals[:1]
        ax.plot(angles, vals, linewidth=1.4, linestyle='solid', label=str(r['Variant']))
        ax.fill(angles, vals, alpha=0.10)
    plt.title(f'{dname}: Fidelity radar (Top-{top_k})');
    plt.legend(bbox_to_anchor=(1.05,1.0), frameon=False)
    plt.tight_layout(); _save(f'radar_{dname}'); plt.show()

# ========= Carga y preprocesado =========
# Ajusta a tu ruta/hoja:
file_path = '/content/drive/MyDrive/transfomer/Resultados.xlsx'
sheet_name = 'newmodels'

# Si estás en Colab, monta Drive (no falla fuera de Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

df = pd.read_excel(file_path, sheet_name=sheet_name)

# Renombrado estándar
rename_map = {
    'method' : 'Variant',
    'dataset': 'Dataset',
    'PCC_mean': 'CC_mean',
    'SpectralCosine_mean': 'SC_mean'
}
df = df.rename(columns=rename_map)

# Conversión numérica segura
num_cols = [
    'RMSE_mean','RMSE_std',
    'CC_mean','PCC_std',
    'SC_mean','SpectralCosine_std',
    'num_params','inference_latency_ms','FLOPs','MACs','Size',
    'RMSE_baseline','y_sigma'
]
for c in num_cols:
    if c in df.columns:
        df[c] = (df[c].astype(str)
                        .str.replace(',', '.', regex=False)
                        .replace({'nan': np.nan, 'None': np.nan}))
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Requisitos mínimos
req = ['Variant','Dataset','RMSE_mean','CC_mean','SC_mean']
for c in req:
    if c not in df.columns:
        raise ValueError(f'Falta columna requerida: {c}')

df = df[(df['RMSE_mean'] >= 0)].copy()
df['CC_mean'] = df['CC_mean'].clip(0, 1)
df['SC_mean'] = df['SC_mean'].clip(0, 1)
if 'Category' in df.columns:
    cat_map = {'cnn':'CNN','rnn':'RNN','hyb':'Hybrid','trasf':'Transformer','mlp':'MLP','other':'Other'}
    df['Type'] = df['Category'].astype(str).str.lower().map(cat_map).fillna('Other')
else:
    df['Type'] = 'Other'

# ========= Cálculo de FI =========
# Si tienes baseline, usa rmse_mode='skill'; si no, cambia a 'nrmse_sigma' o deja fallback.
df_fi = add_fidelity_by_dataset(df, rmse_mode='skill')

# ========= Pipeline por dataset =========
tops_fi_all = []
for dname, g in df_fi.groupby('Dataset', sort=False):
    g = g.copy()
    # 1) Leaderboard FI
    fi_bar_top = plot_fi_top(g, dname, top_k=TOP_K)

    # 2) Scatter con Pareto + labels Top-K
    labels_for_scatter = g.merge(fi_bar_top[['Variant']], on='Variant', how='inner')
    plot_scatter_rmse_pcc(g, dname, labels_for_scatter[['Variant','RMSE_mean','CC_mean']])

    # 3) Barras por métrica
    plot_metric_bars_with_error(g, dname, top_k=TOP_K)

    # 4) Average rank y Heatmap
    _ = plot_avg_rank_bars(g, dname, top_k=TOP_K)
    plot_rank_heatmap(g, dname, top_k=TOP_K)

    # 5) Radar (Top-5 por defecto)
    plot_radar_topk(g, dname, top_k=min(5, TOP_K))

    # Guardar Top-K FI en lista
    top_fi = (g.sort_values('FI', ascending=False)
                .head(TOP_K)
                [['Variant','Type','FI','CC_mean','u_RMSE','SC_mean']])
    top_fi.insert(0, 'Dataset', dname)
    tops_fi_all.append(top_fi)

# ========= CSV agregado con los Top-K FI =========
if tops_fi_all:
    out_csv = os.path.join(OUTPUT_DIR, 'tops_fi_por_dataset.csv')
    pd.concat(tops_fi_all, ignore_index=True).to_csv(out_csv, index=False)
    print(f"\nArchivo guardado: {out_csv}")

print("\nListo. Figuras en:", OUTPUT_DIR)

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ------------- Estilo (paper) -------------
plt.style.use('default')
sns.set_palette("colorblind")
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.figsize': (10, 6),
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.transparent': True
})

# ------------- Parámetros globales -------------
# Normalización robusta por percentiles para evitar outliers
ROBUST = True
Q_LOW, Q_HIGH = 0.05, 0.95

# Pesos de FI (solo similitud)
W_PCC, W_SC, W_RMSE = 0.45, 0.35, 0.20

# Umbrales de factibilidad operativa (ajusta a tus constraints)
LAT_THR = 10.0     # ms
MACS_THR = 1e8
PARAMS_THR = np.inf

# Ranking por dataset
TOP_K = 10

# ------------- Utilidades -------------
def robust_minmax(series, robust=ROBUST, q_low=Q_LOW, q_high=Q_HIGH):
    """Min-max por dataset (opcional robusto). Si sin rango, devuelve 0.5."""
    s = pd.to_numeric(series, errors='coerce').astype(float).copy()
    if robust:
        lo, hi = s.quantile(q_low), s.quantile(q_high)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = s.min(), s.max()
    else:
        lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.full(len(s), 0.5), index=s.index)
    out = (s - lo) / (hi - lo)
    return out.clip(0, 1)

def pareto_mask_minCI_maxFI(df_xy):
    """Máscara Pareto (CI ↓, FI ↑): CI ascendente y mantener FI no-decreciente."""
    d = df_xy.sort_values('CI', ascending=True).reset_index(drop=False)
    keep = np.zeros(len(d), dtype=bool)
    max_fi = -np.inf
    for i, fi in enumerate(d['FI'].values):
        if fi >= max_fi:
            keep[i] = True
            max_fi = fi
    mask = pd.Series(False, index=df_xy.index)
    mask.loc[d.loc[keep, 'index'].values] = True
    return mask

# ------------- Carga de datos -------------
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/UGR/papers/paper estudio comparativo TSER/Resultados.xlsx'
df = pd.read_excel(file_path, sheet_name='newmodels')

# Renombrar columnas estándar
rename_map = {
    'method' : 'Variant',
    'dataset': 'Dataset',
    'PCC_mean': 'CC_mean',
    'SpectralCosine_mean': 'SC_mean'
}
df = df.rename(columns=rename_map)

# Conversión numérica (comas -> punto)
num_cols = [
    'RMSE_mean','RMSE_std',
    'CC_mean','PCC_std',
    'SC_mean','SpectralCosine_std',
    'num_params','inference_latency_ms','FLOPs','MACs','Size'
]
for c in num_cols:
    if c in df.columns:
        df[c] = (df[c].astype(str)
                        .str.replace(',', '.', regex=False)
                        .replace({'nan': np.nan, 'None': np.nan}))
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Sanitiza campos mínimos
req = ['Variant','Dataset','RMSE_mean','CC_mean','SC_mean',
       'inference_latency_ms','MACs','num_params']
for c in req:
    if c not in df.columns:
        raise ValueError(f'Falta columna requerida: {c}')

df = df[(df['RMSE_mean'] >= 0)].copy()
df['CC_mean'] = df['CC_mean'].clip(0, 1)
df['SC_mean'] = df['SC_mean'].clip(0, 1)

# Tipo (si hay Category)
if 'Category' in df.columns:
    cat_map = {'cnn':'CNN','rnn':'RNN','hyb':'Hybrid','trasf':'Transformer','mlp':'MLP','other':'Other'}
    df['Type'] = df['Category'].astype(str).str.lower().map(cat_map).fillna('Other')
else:
    df['Type'] = 'Other'

# ============================================================
# (A) CALCULO DE FIDELIDAD (solo similitud) — por dataset
#     FI = W_PCC*PCC + W_SC*SC + W_RMSE*(1 - RMSE_norm_dataset)
# ============================================================
def add_fidelity_by_dataset(frame: pd.DataFrame) -> pd.DataFrame:
    out = []
    for dname, g in frame.groupby('Dataset', sort=False):
        g = g.copy()
        g['RMSE_n'] = robust_minmax(g['RMSE_mean'], robust=True)  # [0,1], menor mejor -> 1 - RMSE_n
        g['FI'] = W_PCC*g['CC_mean'] + W_SC*g['SC_mean'] + W_RMSE*(1 - g['RMSE_n'])
        out.append(g)
    return pd.concat(out, ignore_index=True)

# ============================================================
# (B) CALCULO DE COMPLEJIDAD — por dataset
#     CI = wL*minmax(log10(lat)) + wM*minmax(log10(MACs)) + wP*minmax(log10(params))
#     Pesos adaptativos si Spearman(lat, MACs) > 0.85
# ============================================================
def add_complexity_by_dataset(frame: pd.DataFrame) -> pd.DataFrame:
    out = []
    for dname, g in frame.groupby('Dataset', sort=False):
        g = g.copy()
        g = g[(g['inference_latency_ms'] > 0) & (g['MACs'] > 0) & (g['num_params'] > 0)]
        if g.empty:
            continue
        rho = g[['inference_latency_ms','MACs']].corr(method='spearman').iloc[0,1]
        if rho > 0.85:
            wL, wM, wP = 0.70, 0.15, 0.15
        else:
            wL, wM, wP = 0.50, 0.30, 0.20
        g['logL']   = np.log10(g['inference_latency_ms'])
        g['logMAC'] = np.log10(g['MACs'])
        g['logP']   = np.log10(g['num_params'])
        g['logL_n']   = robust_minmax(g['logL'], robust=True)
        g['logMAC_n'] = robust_minmax(g['logMAC'], robust=True)
        g['logP_n']   = robust_minmax(g['logP'], robust=True)
        g['CI'] = wL*g['logL_n'] + wM*g['logMAC_n'] + wP*g['logP_n']
        g['wL'], g['wM'], g['wP'] = wL, wM, wP
        out.append(g)
    return pd.concat(out, ignore_index=True)

# ============================================================
# (C) RELACION FI–CI — factibilidad, Pareto, rankings, gráficas
# ============================================================
def add_feasibility_and_pareto(frame: pd.DataFrame) -> pd.DataFrame:
    out = []
    for dname, g in frame.groupby('Dataset', sort=False):
        g = g.copy()
        g['Feasible'] = (
            (g['inference_latency_ms'] <= LAT_THR) &
            (g['MACs'] <= MACS_THR) &
            (g['num_params'] <= PARAMS_THR)
        )
        g['Pareto_2D'] = pareto_mask_minCI_maxFI(g[['FI','CI']])
        out.append(g)
    return pd.concat(out, ignore_index=True)

# ---------------------- PLOTS ----------------------
def plot_fi_top(g, dname, top_k=TOP_K):
    top = (g.sort_values('FI', ascending=False)
             .head(top_k)
             [['Variant','Type','FI','CI','Feasible','Pareto_2D']]
             .reset_index(drop=True))
    # barras
    plt.figure(figsize=(10, 5.2))
    order = top.sort_values('FI', ascending=True)
    colors = ['#2ca02c' if (p and f) else '#1f77b4' if f else '#7f7f7f'
              for p, f in zip(order['Pareto_2D'], order['Feasible'])]
    plt.barh(order['Variant'], order['FI'], color=colors, alpha=0.9)
    plt.xlabel('Fidelity Index (per-dataset, 0–1)')
    plt.title(f'{dname}: Top-{top_k} Fidelity Index')
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout(); plt.savefig(f'fi_top_{dname}.png', dpi=300); plt.show()
    return top

def plot_ci_top(g, dname, top_k=TOP_K):
    top = (g.sort_values('CI', ascending=True)
             .head(top_k)
             [['Variant','Type','FI','CI','Feasible','Pareto_2D','logL_n','logMAC_n','logP_n','wL','wM','wP']]
             .reset_index(drop=True))
    # barras (CI: menor mejor)
    plt.figure(figsize=(10, 5.2))
    order = top.sort_values('CI', ascending=True)
    colors = ['#2ca02c' if (p and f) else '#1f77b4' if f else '#7f7f7f'
              for p, f in zip(order['Pareto_2D'], order['Feasible'])]
    plt.barh(order['Variant'], order['CI'], color=colors, alpha=0.9)
    plt.xlabel('Complexity Index (per-dataset, 0–1; lower is better)')
    plt.title(f'{dname}: Top-{top_k} Complexity Index')
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout(); plt.savefig(f'ci_top_{dname}.png', dpi=300); plt.show()
    return top

def plot_ci_components(top_ci, dname):
    """Desglose de componentes normalizados del CI (logL_n, logMAC_n, logP_n) como barras apiladas."""
    if top_ci.empty: return
    comp = top_ci[['Variant','logL_n','logMAC_n','logP_n']].set_index('Variant')
    plt.figure(figsize=(10.5, 5.6))
    bottom = np.zeros(len(comp))
    for col, label in zip(['logL_n','logMAC_n','logP_n'], ['Latency','MACs','Params']):
        plt.bar(comp.index, comp[col], bottom=bottom, label=label, alpha=0.9)
        bottom += comp[col].values
    plt.ylabel('Normalized components (0–1)')
    plt.xticks(rotation=45, ha='right')
    plt.title(f'{dname}: CI components for Top-{len(comp)} (lower total is better)')
    plt.legend(frameon=True)
    plt.tight_layout(); plt.savefig(f'ci_components_{dname}.png', dpi=300); plt.show()

def plot_scatter_rmse_pcc(g, dname, top_labels):
    plt.figure(figsize=(8.6, 6.2))
    sc = plt.scatter(g['RMSE_mean'], g['CC_mean'],
                     s=300*g['SC_mean'].fillna(0.5)+30,
                     c=g['FI'], cmap='viridis', alpha=0.9,
                     edgecolors='k', linewidths=0.3)
    plt.colorbar(sc, label='Fidelity Index (per-dataset)')
    plt.xlabel('RMSE (lower is better)')
    plt.ylabel('PCC (higher is better)')
    plt.title(f'{dname}: RMSE vs PCC (bubble ~ SC, color ~ FI)')
    for _, r in top_labels.iterrows():
        plt.annotate(str(r['Variant']),
                     (float(r['RMSE_mean']), float(r['CC_mean'])),
                     fontsize=9, xytext=(5,5), textcoords='offset points')
    plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(f'scatter_{dname}.png', dpi=300); plt.show()

def plot_pareto(g, dname):
    plt.figure(figsize=(9.2, 6.2))
    plt.scatter(g['CI'], g['FI'], s=30, c='0.75', label='All models', zorder=1)
    p_all = g[g['Pareto_2D']]
    if not p_all.empty:
        plt.scatter(p_all['CI'], p_all['FI'], facecolors='none', edgecolors='k',
                    s=80, linewidths=1.2, label='Pareto (FI–CI)', zorder=3)
    gf = g[g['Feasible']]
    if not gf.empty:
        mask_pf = pareto_mask_minCI_maxFI(gf[['FI','CI']])
        pf = gf[mask_pf]
        if not pf.empty:
            plt.scatter(pf['CI'], pf['FI'], s=60, c='#2ca02c',
                        label='Feasible Pareto', zorder=4)
    plt.xlabel('Complexity Index (lower is better)')
    plt.ylabel('Fidelity Index (higher is better)')
    plt.title(f'{dname}: Fidelity–Complexity Pareto View')
    plt.grid(True, alpha=0.3); plt.legend(loc='lower left', frameon=True)
    plt.tight_layout(); plt.savefig(f'pareto_{dname}.png', dpi=300); plt.show()

# ---------------------- REPORTES ----------------------
def report_and_plot_by_dataset(frame: pd.DataFrame, top_k=TOP_K):
    tops_fi_all, tops_ci_all, pareto_feasible_all = [], [], []

    for dname, g in frame.groupby('Dataset', sort=False):
        g = g.copy()

        # ------- TOP-K por FI -------
        top_fi = (g.sort_values('FI', ascending=False)
                    .head(top_k)
                    [['Variant','Type','FI','CI','CC_mean','RMSE_mean','SC_mean',
                      'inference_latency_ms','MACs','num_params','Feasible','Pareto_2D']])
        top_fi.insert(0, 'Dataset', dname)
        tops_fi_all.append(top_fi)

        print(f"\n=== TOP-{top_k} por FI — Dataset: {dname} ===")
        print(top_fi.round(4).to_string(index=False))

        # ------- TOP-K por CI (menor mejor) -------
        top_ci = (g.sort_values('CI', ascending=True)
                    .head(top_k)
                    [['Variant','Type','FI','CI','CC_mean','RMSE_mean','SC_mean',
                      'inference_latency_ms','MACs','num_params','Feasible','Pareto_2D',
                      'logL_n','logMAC_n','logP_n','wL','wM','wP']])
        top_ci.insert(0, 'Dataset', dname)
        tops_ci_all.append(top_ci)

        print(f"\n=== TOP-{top_k} por CI — Dataset: {dname} (lower is better) ===")
        print(top_ci.round(4).to_string(index=False))

        # ------- Pareto factible (con umbrales) -------
        gf = g[g['Feasible']]
        if not gf.empty:
            mask_pf = pareto_mask_minCI_maxFI(gf[['FI','CI']])
            pf = gf[mask_pf].copy()
            if not pf.empty:
                sel = pf[['Variant','Type','FI','CI','CC_mean','RMSE_mean','SC_mean',
                          'inference_latency_ms','MACs','num_params']]
                sel.insert(0, 'Dataset', dname)
                pareto_feasible_all.append(sel)

        # ------- Figuras -------
        fi_bar_top = plot_fi_top(g, dname, top_k=top_k)
        ci_bar_top = plot_ci_top(g, dname, top_k=top_k)
        plot_ci_components(ci_bar_top, dname)
        # Usa los del TOP FI para etiquetar en scatter
        labels_for_scatter = g.merge(fi_bar_top[['Variant']], on='Variant', how='inner')
        plot_scatter_rmse_pcc(g, dname, labels_for_scatter)
        plot_pareto(g, dname)

    # Guardar CSVs agregados
    if tops_fi_all:
        pd.concat(tops_fi_all, ignore_index=True).to_csv('tops_fi_por_dataset.csv', index=False)
        print("\nArchivo guardado: tops_fi_por_dataset.csv")
    if tops_ci_all:
        pd.concat(tops_ci_all, ignore_index=True).to_csv('tops_ci_por_dataset.csv', index=False)
        print("Archivo guardado: tops_ci_por_dataset.csv")
    if pareto_feasible_all:
        pd.concat(pareto_feasible_all, ignore_index=True).to_csv('pareto_factible_por_dataset.csv', index=False)
        print("Archivo guardado: pareto_factible_por_dataset.csv")

# ================== PIPELINE ==================
# A) Fidelidad
df_fi = add_fidelity_by_dataset(df)

# B) Complejidad
df_fi_ci = add_complexity_by_dataset(df_fi)

# C) Relación FI–CI (factibilidad + Pareto) y reportes
df_final = add_feasibility_and_pareto(df_fi_ci)
report_and_plot_by_dataset(df_final, top_k=TOP_K)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/transfomer/Resultados.xlsx'

In [ ]:
from google.colab import files

# CSVs
files.download('tops_fi_por_dataset.csv')
files.download('tops_ci_por_dataset.csv')
files.download('pareto_factible_por_dataset.csv')

# Figuras de un dataset concreto (cambia el nombre del dataset)
files.download('fi_top_utah2002.png')
files.download('ci_top_utah2002.png')
files.download('ci_components_utah2002.png')
files.download('scatter_utah2002.png')
files.download('pareto_utah2002.png')


In [ ]:
import glob, zipfile
from google.colab import files

artifacts = glob.glob('*.png') + glob.glob('*.csv')
zip_name = 'outputs_all.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in artifacts:
        z.write(f)

files.download(zip_name)


In [ ]:
# ============================================================
# FIDELIDAD-SOLO (paper)
# Figuras por dataset a partir de: RMSE_mean, CC_mean, SC_mean (+ std opcional)
# - FI = 0.45*PCC + 0.35*SC + 0.20*(1 - RMSE_norm_dataset)
# - Top-K por FI, dispersion RMSE–PCC (burbuja SC, color FI)
# - Barras con error por métrica (RMSE, PCC, SC)
# - Average rank y Heatmap de rangos
# - Radar para Top-K
# Salidas: PNGs por dataset + CSV de Top-K FI
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ------------- Estilo (paper) -------------
plt.style.use('default')
sns.set_palette("colorblind")
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.figsize': (10, 6),
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.transparent': True
})

# ------------- Parámetros -------------
ROBUST = True
Q_LOW, Q_HIGH = 0.05, 0.95
W_PCC, W_SC, W_RMSE = 0.45, 0.35, 0.20
TOP_K = 10

# ==== UTILIDADES ====
def robust_minmax(series, robust=ROBUST, q_low=Q_LOW, q_high=Q_HIGH):
    """Min-max por dataset (opcional robusto). Si sin rango, devuelve 0.5."""
    s = pd.to_numeric(series, errors='coerce').astype(float).copy()
    if robust:
        lo, hi = s.quantile(q_low), s.quantile(q_high)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = s.min(), s.max()
    else:
        lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.full(len(s), 0.5), index=s.index)
    out = (s - lo) / (hi - lo)
    return out.clip(0, 1)

def rank_block_by_dataset(g: pd.DataFrame) -> pd.DataFrame:
    """Rangos por métrica dentro del dataset (1 = mejor)."""
    out = g.copy()
    out['rank_RMSE'] = out['RMSE_mean'].rank(ascending=True,  method='min')
    out['rank_PCC']  = out['CC_mean'].rank(ascending=False, method='min')
    out['rank_SC']   = out['SC_mean'].rank(ascending=False, method='min')
    out['avg_rank']  = out[['rank_RMSE','rank_PCC','rank_SC']].mean(axis=1)
    return out

# ==== CARGA Y PREPROCESADO ====
# Cambia esta ruta/hoja a las tuyas:
file_path = '/content/drive/MyDrive/UGR/papers/paper estudio comparativo TSER/Resultados.xlsx'
sheet_name = 'newmodels'

# Si estás en Colab, monta Drive; si no, comenta estas 2 líneas:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

df = pd.read_excel(file_path, sheet_name=sheet_name)

# Renombrar columnas estándar esperadas
rename_map = {
    'method' : 'Variant',
    'dataset': 'Dataset',
    'PCC_mean': 'CC_mean',
    'SpectralCosine_mean': 'SC_mean'
}
df = df.rename(columns=rename_map)

# Conversión numérica (comas -> punto) en estas columnas si existen
num_cols = [
    'RMSE_mean','RMSE_std',
    'CC_mean','PCC_std',
    'SC_mean','SpectralCosine_std',
    'num_params','inference_latency_ms','FLOPs','MACs','Size'
]
for c in num_cols:
    if c in df.columns:
        df[c] = (df[c].astype(str)
                        .str.replace(',', '.', regex=False)
                        .replace({'nan': np.nan, 'None': np.nan}))
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Sanitizado mínimo de columnas necesarias para fidelidad
req = ['Variant','Dataset','RMSE_mean','CC_mean','SC_mean']
for c in req:
    if c not in df.columns:
        raise ValueError(f'Falta columna requerida: {c}')

df = df[(df['RMSE_mean'] >= 0)].copy()
df['CC_mean'] = df['CC_mean'].clip(0, 1)
df['SC_mean'] = df['SC_mean'].clip(0, 1)

# Tipo (si hubiera Category)
if 'Category' in df.columns:
    cat_map = {'cnn':'CNN','rnn':'RNN','hyb':'Hybrid','trasf':'Transformer','mlp':'MLP','other':'Other'}
    df['Type'] = df['Category'].astype(str).str.lower().map(cat_map).fillna('Other')
else:
    df['Type'] = 'Other'

# ==== FIDELIDAD ====
def add_fidelity_by_dataset(frame: pd.DataFrame) -> pd.DataFrame:
    """
    FI por dataset:
      FI = 0.45*PCC + 0.35*SC + 0.20*(1 - RMSE_norm_dataset)
    """
    out = []
    for dname, g in frame.groupby('Dataset', sort=False):
        g = g.copy()
        g['RMSE_n'] = robust_minmax(g['RMSE_mean'], robust=True)  # [0,1], menor mejor -> 1 - RMSE_n
        g['FI'] = W_PCC*g['CC_mean'] + W_SC*g['SC_mean'] + W_RMSE*(1 - g['RMSE_n'])
        out.append(g)
    return pd.concat(out, ignore_index=True)

# ==== PLOTS (FIDELIDAD) ====
def plot_fi_top(g, dname, top_k=TOP_K):
    """Leaderboard de FI — Top-K."""
    top = (g.sort_values('FI', ascending=False)
             .head(top_k)
             [['Variant','Type','FI','CC_mean','RMSE_mean','SC_mean']]
             .reset_index(drop=True))
    plt.figure(figsize=(10, 5.2))
    order = top.sort_values('FI', ascending=True)
    plt.barh(order['Variant'], order['FI'], color='#1f77b4', alpha=0.9)
    plt.xlabel('Fidelity Index (per-dataset, 0–1)')
    plt.title(f'{dname}: Top-{top_k} Fidelity Index')
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout(); plt.savefig(f'fi_top_{dname}.png'); plt.show()
    return top

def plot_scatter_rmse_pcc(g, dname, top_labels):
    """Dispersión RMSE vs PCC; tamaño ~ SC, color ~ FI. Etiqueta los Top-K por FI."""
    plt.figure(figsize=(8.6, 6.2))
    sc = plt.scatter(g['RMSE_mean'], g['CC_mean'],
                     s=300*g['SC_mean'].fillna(0.5)+30,
                     c=g['FI'], cmap='viridis', alpha=0.9,
                     edgecolors='k', linewidths=0.3)
    plt.colorbar(sc, label='Fidelity Index (per-dataset)')
    plt.xlabel('RMSE (lower is better)')
    plt.ylabel('PCC (higher is better)')
    plt.title(f'{dname}: RMSE vs PCC (bubble ~ SC, color ~ FI)')
    for _, r in top_labels.iterrows():
        plt.annotate(str(r['Variant']),
                     (float(r['RMSE_mean']), float(r['CC_mean']))),
    plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(f'scatter_{dname}.png'); plt.show()

def plot_metric_bars_with_error(g, dname, top_k=TOP_K):
    """Barras con error para RMSE, PCC y SC (Top-K por FI)."""
    top = g.sort_values('FI', ascending=False).head(top_k)
    order = top.sort_values('RMSE_mean', ascending=True)['Variant']  # ordena por RMSE

    # RMSE
    plt.figure(figsize=(10,5))
    y = top.set_index('Variant').loc[order]['RMSE_mean']
    yerr = top.set_index('Variant').loc[order]['RMSE_std'] if 'RMSE_std' in top else None
    plt.barh(y.index, y.values, xerr=None if (yerr is None or yerr.isna().all()) else yerr.values, alpha=0.9)
    plt.xlabel('RMSE (lower is better)'); plt.title(f'{dname}: RMSE (Top-{top_k})')
    plt.grid(True, axis='x', alpha=0.3); plt.tight_layout(); plt.savefig(f'rmse_bars_{dname}.png'); plt.show()

    # PCC
    plt.figure(figsize=(10,5))
    y = top.set_index('Variant').loc[order]['CC_mean']
    yerr = top.set_index('Variant').loc[order]['PCC_std'] if 'PCC_std' in top else None
    plt.barh(y.index, y.values, xerr=None if (yerr is None or yerr.isna().all()) else yerr.values, alpha=0.9)
    plt.xlabel('PCC (higher is better)'); plt.title(f'{dname}: PCC (Top-{top_k})')
    plt.grid(True, axis='x', alpha=0.3); plt.tight_layout(); plt.savefig(f'pcc_bars_{dname}.png'); plt.show()

    # SC
    plt.figure(figsize=(10,5))
    y = top.set_index('Variant').loc[order]['SC_mean']
    yerr = top.set_index('Variant').loc[order]['SpectralCosine_std'] if 'SpectralCosine_std' in top else None
    plt.barh(y.index, y.values, xerr=None if (yerr is None or yerr.isna().all()) else yerr.values, alpha=0.9)
    plt.xlabel('SC (higher is better)'); plt.title(f'{dname}: SC (Top-{top_k})')
    plt.grid(True, axis='x', alpha=0.3); plt.tight_layout(); plt.savefig(f'sc_bars_{dname}.png'); plt.show()

def plot_avg_rank_bars(g, dname, top_k=TOP_K):
    """Barras de promedio de rangos (menor = mejor)."""
    gr = rank_block_by_dataset(g).sort_values('avg_rank', ascending=True).head(top_k)
    plt.figure(figsize=(10,5))
    plt.barh(gr['Variant'], gr['avg_rank'], alpha=0.9)
    plt.xlabel('Average rank (RMSE, PCC, SC)'); plt.title(f'{dname}: Average rank (Top-{top_k})')
    plt.gca().invert_yaxis()
    plt.grid(True, axis='x', alpha=0.3); plt.tight_layout(); plt.savefig(f'avg_rank_{dname}.png'); plt.show()
    return gr[['Variant','avg_rank','rank_RMSE','rank_PCC','rank_SC']]

def plot_rank_heatmap(g, dname, top_k=TOP_K):
    """Heatmap de rangos por métrica para los Top-K por FI."""
    import numpy as np
    top = g.sort_values('FI', ascending=False).head(top_k)
    ranks = rank_block_by_dataset(top)[['Variant','rank_RMSE','rank_PCC','rank_SC']].set_index('Variant')
    plt.figure(figsize=(6, 0.5 + 0.35*len(ranks)))
    plt.imshow(ranks.values, aspect='auto', cmap='viridis_r')
    plt.colorbar(label='Rank (1=best)')
    plt.xticks(np.arange(3), ['RMSE','PCC','SC'])
    plt.yticks(np.arange(len(ranks.index)), ranks.index)
    plt.title(f'{dname}: Metric ranks (Top-{top_k} by FI)')
    plt.tight_layout(); plt.savefig(f'rank_heatmap_{dname}.png'); plt.show()

def plot_radar_topk(g, dname, top_k=5):
    """Radar de fidelidad: [PCC, SC, 1-RMSE_n] para Top-K por FI."""
    from math import pi
    gg = g.copy()
    gg['RMSE_n'] = robust_minmax(gg['RMSE_mean'], robust=True)
    gg['ONE_mRMSE'] = 1 - gg['RMSE_n']
    top = gg.sort_values('FI', ascending=False).head(top_k)
    metrics = ['CC_mean','SC_mean','ONE_mRMSE']
    labels = ['PCC','SC','1 - RMSE_n']
    angles = [n / float(len(metrics)) * 2 * pi for n in range(len(metrics))]
    angles += angles[:1]
    plt.figure(figsize=(6.5,6.5))
    ax = plt.subplot(111, polar=True)
    ax.set_theta_offset(pi / 2); ax.set_theta_direction(-1)
    plt.xticks(angles[:-1], labels)
    ax.set_rlabel_position(0); plt.yticks([0.25,0.5,0.75], ['0.25','0.5','0.75'], color="grey", size=9); plt.ylim(0,1)
    for _, r in top.iterrows():
        vals = [float(r[m]) for m in metrics]; vals += vals[:1]
        ax.plot(angles, vals, linewidth=1, linestyle='solid', label=str(r['Variant']))
        ax.fill(angles, vals, alpha=0.10)
    plt.title(f'{dname}: Fidelity radar (Top-{top_k})'); plt.legend(bbox_to_anchor=(1.05,1.0))
    plt.tight_layout(); plt.savefig(f'radar_{dname}.png'); plt.show()

# ==== PIPELINE: FIDELIDAD-SOLO ====
df_fi = add_fidelity_by_dataset(df)

# CSV de Top-K FI por dataset (opcional)
tops_fi_all = []

for dname, g in df_fi.groupby('Dataset', sort=False):
    g = g.copy()

    # 1) Leaderboard FI
    fi_bar_top = plot_fi_top(g, dname, top_k=TOP_K)

    # 2) Dispersión RMSE–PCC (burbuja SC, color FI) con labels del Top-K FI
    labels_for_scatter = g.merge(fi_bar_top[['Variant']], on='Variant', how='inner')
    plot_scatter_rmse_pcc(g, dname, labels_for_scatter[['Variant','RMSE_mean','CC_mean']])

    # 3) Barras con error por métrica
    plot_metric_bars_with_error(g, dname, top_k=TOP_K)

    # 4) Average rank y Heatmap de rangos
    _ = plot_avg_rank_bars(g, dname, top_k=TOP_K)
    plot_rank_heatmap(g, dname, top_k=TOP_K)

    # 5) Radar (Top-5 por defecto)
    plot_radar_topk(g, dname, top_k=min(5, TOP_K))

    # guardar Top-K FI a una lista
    top_fi = (g.sort_values('FI', ascending=False)
                .head(TOP_K)
                [['Variant','Type','FI','CC_mean','RMSE_mean','SC_mean']])
    top_fi.insert(0, 'Dataset', dname)
    tops_fi_all.append(top_fi)

# Guardar CSV agregado con los Top-K FI de todos los datasets
if tops_fi_all:
    pd.concat(tops_fi_all, ignore_index=True).to_csv('tops_fi_por_dataset.csv', index=False)
    print("\nArchivo guardado: tops_fi_por_dataset.csv")


Output hidden; open in https://colab.research.google.com to view.